# Seguimiento de características (tracking)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oscar-ramos/robotica-autonoma-python/blob/main/3-Vision/3-3-CV-Tradicional/3-3-4-Seguimiento.ipynb)

Se detecta puntos "buenos para seguir" usando Shi-Tomasi. Luego, entre dos imágenes consecutivas se estima el desplazamiento local $(u, v)$ de cada punto mediante Lucas-Kanade.

Se utiliza la ecuación de flujo óptico $I_x u + I_y v + I_t = 0$. Como una sola ecuación no basta para hallar u y v, Lucas-Kanade asume que todos los píxeles dentro de una ventana pequeña $W$ tienen el mismo movimiento. Entonces resuelve el sistema local por mínimos cuadrados. En OpenCV, esto se implementa con `calcOpticalFlowPyrLK()`. 

In [1]:
import cv2
import numpy as np

### Lectura de Video

In [ ]:
colab = True
if (colab):
    # Se usa un video que proviene de https://cvg.cit.tum.de/data/datasets/rgbd-dataset/download
    !wget -q https://github.com/oscar-ramos/robotica-autonoma-python/raw/main/3-Vision/3-3-CV-Tradicional/imagenes/freiburg1_short.mp4

In [ ]:
# Lectura de video (0 para cámara)
cap = cv2.VideoCapture("freiburg1_short.mp4")

if not cap.isOpened(): # si no se puede abrir el video
    raise RuntimeError("No se pudo abrir el video.")

# Lectura del primer frame
ret, old_frame = cap.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame.")

### Parámetros

In [4]:
# Colores aleatorios para visualizar trayectorias
rng = np.random.default_rng(42)
colors = rng.integers(0, 255, size=(500, 3), dtype=np.uint8)

# Parámetros para Shi-Tomasi: goodFeaturesToTrack() detecta puntos con buena estructura local.
feature_params = dict(
    maxCorners = 200,       # máximo número de puntos a detectar
    qualityLevel = 0.01,    # umbral de calidad relativo
    minDistance = 7,        # distancia mínima entre puntos
    blockSize = 7           # tamaño de ventana para análisis local
)

# Parámetros para Lucas-Kanade piramidal
lk_params = dict(
    winSize=(21, 21),   # ventana local W donde se asume movimiento constante
    maxLevel=3,         # número de niveles en la pirámide
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)  # criterio de parada
)

# Contador para reinyectar puntos si se pierden demasiados
frame_idx = 0
reinjection_interval = 30   # cada cuántos frames considerar redetección
min_points_threshold = 50  

# Video de salida
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('video_salida.mp4', fourcc, 20.0, (old_frame.shape[1], old_frame.shape[0]))

In [ ]:
# Conversión a gris (el flujo óptico trabaja sobre intensidad)
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Detección de puntos iniciales con Shi-Tomasi
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

while True:
    # Lectura de un nuevo frame del video
    ret, frame = cap.read()
    if not ret:
        break

    # Convertir el frame a escala de grises
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Crear una copia del frame para dibujar trayectorias recientes 
    display = frame.copy()

    # Lucas-Kanade: nueva posición de los puntos p0 en el frame actual.
    #   p1: posiciones estimadas en el nuevo frame
    #   st: indica si el punto fue seguido exitosamente (punto válido)
    #   err: error estimado del seguimiento
    if p0 is not None and len(p0) > 0:
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        # Selección de puntos válidos
        if p1 is not None and st is not None:
            good_new = p1[st.flatten() == 1]
            good_old = p0[st.flatten() == 1]

            # Dibujar trajectorias
            for i, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel()
                c, d = old.ravel()
                color = tuple(int(x) for x in colors[i % len(colors)])

                # Línea desde posición anterior a nueva
                # cv2.line(display, (int(c), int(d)), (int(a), int(b)), color, 2)
                cv2.line(display, (int(c), int(d)), (int(c + 4*(a-c)), int(d + 4*(b-d))), color, 1, cv2.LINE_AA)

                # Círculo pequeño en la posición actual del feature
                cv2.circle(display, (int(a), int(b)), 2, color, -1, cv2.LINE_AA)

            # Mostrar la cantidad de puntos seguidos
            cv2.putText(display, f"Puntos seguidos: {len(good_new)}", (20, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            # Cálculo del flujo medio
            if len(good_new) > 0:
                displacements = good_new - good_old
                mean_disp = np.mean(displacements, axis=0)
                dx, dy = mean_disp.ravel()
                cv2.putText(display, f"Flujo medio: dx={dx:.2f}, dy={dy:.2f}", (20, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

            # Mostrar la imagen con el tracking
            if not colab:
                cv2.imshow("Tracking de caracteristicas", display)
            out.write(display)

            # Actualizar el frame anterior y los puntos
            old_gray = frame_gray.copy()
            p0 = good_new.reshape(-1, 1, 2)

        else:
            # Si no se pudieron seguir los puntos, detectar nuevos
            p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
            old_gray = frame_gray.copy()
            if not colab:
                cv2.imshow("Tracking de caracteristicas", display)
            out.write(display)

    else:
        # Si no hay puntos, detectar nuevos
        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
        old_gray = frame_gray.copy()
        if not colab:
            cv2.imshow("Tracking de caracteristicas", display)
        out.write(display)

    # Reinyección de puntos cada cierto número de frames
    frame_idx += 1
    if frame_idx % reinjection_interval == 0:
        # Si no hay suficientes puntos, detectar nuevos
        if p0 is None or len(p0) < min_points_threshold:
            # Detectar nuevos puntos
            new_points = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
            # Si se detectaron nuevos puntos, agregarlos a la lista de puntos
            if new_points is not None:
                if p0 is None:
                    p0 = new_points
                else:
                    p0 = np.vstack((p0, new_points))

    # Salir del bucle si se presiona la tecla ESC o 'q'
    if not colab:
        key = cv2.waitKey(30) & 0xFF
        if key == 27 or key == ord('q'):
            break

# Liberar recursos
cap.release()
if not colab:
    cv2.destroyAllWindows()
out.release()

In [ ]:
# En Colab:
from IPython.display import Video
Video("output.mp4")